## Before You Start: Key Project Hints

This dataset mixes movie-level tables and rating-event-level data. The key skill is controlling granularity.

- A common final ABT is **one row per movie**.
- Do **not** directly merge raw `ratings` into `movies` and assume the result is still one row per movie.
- Aggregate ratings to `movieId` first, then merge the rating summary into the movie table.
- Use `movieId` for ratings and `tmdbId` for curated credits/keywords.
- Average rating is unstable for movies with very few ratings. Use a minimum rating-count threshold before modeling.
- Be careful about leakage: post-release fields such as revenue, popularity, vote count, or rating count may not be valid if your question is pre-release prediction.

## Setup

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists() and (REPO_ROOT.parent / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
REPORTS_DIR = REPO_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)


### Key and Granularity Audit

Use the expected key for each table before planning joins. `ratings` is the exception: its row meaning is a user-movie rating event, so check the combined `userId`-`movieId` key rather than expecting `movieId` alone to be unique. `links` is a crosswalk; because `movies` already contains both IDs, use it for validation unless your analysis needs one of its fields.


## Milestone 1 Starter: Initial Profile and Cleaning Plan

Important Movies notes:

- `movies` is movie-level, while `ratings` is user-movie rating-event-level.
- `credits` and `keywords` are curated movie-level tables connected by `tmdbId`.
- Budget and revenue may contain zero or missing-like values; describe this before using them.
- Decide whether your project is about audience ratings, catalog strategy, revenue, popularity, or genre/content patterns.

## Business Problem and Project Framing
Many movies are included in the catalogs of streaming platforms. These platforms need to understand what characteristics are associated with higher audience ratings. Understanding these patterns can help streaming companies guide informed decisons about their available movie content.

The intended user is the streaming platforms content and marketing teams. The teams could use this analysis to understand which movies receive higher ratings when choosing which movies to add or promote on their platform.

Analytical Question: What movie characteristics are associated with higher ratings?


## Data Description and Row Meaning

The dataset comes from the BAN6003 Project Option 1 Github repositiory.
https://github.com/tianhaiz/ban6003-project-option-1-movies-template
 This dataset includes 5 raw CSV files. 
 
 Below is futher information of each file, table name, what it contains and meaning of each row. 

movies_metadata.csv , 'movies' , general information on movies, 
one row = one movie.

ratings.csv , 'ratings' , what a user rates a movie, 
one row = one users rating of one movie.

links.csv , 'links' , connects movie ids , IMDb ids, and TMBD ids.
one row = one movie

credits_curated.csv, 'credits' , consists of director and main case names.
one row = one movie

keywords_curated.csv, 'keywords' , keywords to describe a movie
one row = one movie



In [3]:
movies = pd.read_csv(DATA_DIR / 'movies_metadata.csv')
ratings = pd.read_csv(DATA_DIR / 'ratings.csv', parse_dates=['rating_datetime'])
links = pd.read_csv(DATA_DIR / 'links.csv')
credits = pd.read_csv(DATA_DIR / 'credits_curated.csv')
keywords = pd.read_csv(DATA_DIR / 'keywords_curated.csv')

tables = {
    'movies': movies,
    'ratings': ratings,
    'links': links,
    'credits': credits,
    'keywords': keywords,
}

In [4]:
for name, df in tables.items():
    print(f'\n{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
    display(df.head(3))


movies: 9,082 rows x 21 columns


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000,373554033.0,21.946943,7.7,5415.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Comedy|Family,Animation,1995
1,2,8844,tt0113497,Jumanji,Jumanji,en,1995-12-15,104.0,65000000,262797249.0,17.015539,6.9,2413.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Fantasy|Family,Adventure,1995
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,101.0,0,0.0,11.712900,6.5,92.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Romance|Comedy,Romance,1995



ratings: 99,810 rows x 5 columns


,userId,movieId,rating,timestamp,rating_datetime
0,1,31,2.5,1260759144,2009-12-14 02:52:24
1,1,1029,3.0,1260759179,2009-12-14 02:52:59
2,1,1061,3.0,1260759182,2009-12-14 02:53:02



links: 9,125 rows x 3 columns


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0



credits: 9,082 rows x 3 columns


,tmdbId,director,top_cast
0,862,John Lasseter,Tom Hanks|Tim Allen|Don Rickles
1,8844,Joe Johnston,Robin Williams|Jonathan Hyde|Kirsten Dunst
2,15602,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret



keywords: 9,082 rows x 2 columns


,tmdbId,keywords_list
0,862,jealousy|toy|boy|friendship|friends|rivalry|bo...
1,8844,board game|disappearance|based on children's b...
2,15602,fishing|best friend|duringcreditsstinger|old men


## Initial Data Profile 

The movies table has 9082 rows and 21 columns, rating table has 99810 rows and 5 columns, links table has 9125 rows and 3 columns, credits table has 9082 rows and 3 columns, and keywords has 9082 rows and 2 columns. No duplicated cells in any of the 5 tables, but missing cells were found in movies, links, credits, and keywords tables.

In [5]:
profile_rows = []
for name, df in tables.items():
    profile_rows.append({
        'table': name,
        'rows': len(df),
        'columns': df.shape[1],
        'duplicate_rows': int(df.duplicated().sum()),
        'missing_cells': int(df.isna().sum().sum()),
    })
pd.DataFrame(profile_rows)

,table,rows,columns,duplicate_rows,missing_cells
0,movies,9082,21,0,72
1,ratings,99810,5,0,0
2,links,9125,3,0,13
3,credits,9082,3,0,114
4,keywords,9082,2,0,761


## Missing Values
The movies table has 35 missing values in genres_list, 35 in primary_genre, and 2 in status. The links table has 13 missing values in tmbID. The credits table has 24 missing director values and 90 missing top_cast values. The keywords table has 761 misssing values i n the keywords list. The ratings table has no missing values.

In [6]:
for name, df in tables.items():
    print(name)
    print(df.isna().sum())
    print()

movies
movieId                  0
tmdbId                   0
imdb_id                  0
title                    0
original_title           0
original_language        0
release_date             0
runtime                  0
budget                   0
revenue                  0
popularity               0
vote_average             0
vote_count               0
genres                   0
production_companies     0
production_countries     0
adult                    0
status                   2
genres_list             35
primary_genre           35
release_year             0
dtype: int64

ratings
userId             0
movieId            0
rating             0
timestamp          0
rating_datetime    0
dtype: int64

links
movieId     0
imdbId      0
tmdbId     13
dtype: int64

credits
tmdbId       0
director    24
top_cast    90
dtype: int64

keywords
tmdbId             0
keywords_list    761
dtype: int64



## Date Types
The data set contained many different data types. Interger values used for whole number movie IDs, and User IDs. Float values used for cells that contain decimals such as ratings, or revenue. String values used for cells with text such as movie names, directors, or ratings. The adult column uses a boolean value while rating_datetime used date and time.

In [7]:
for name, df in tables.items():
    print(name)
    print(df.dtypes)

movies
movieId                   int64
tmdbId                    int64
imdb_id                     str
title                       str
original_title              str
original_language           str
release_date                str
runtime                 float64
budget                    int64
revenue                 float64
popularity              float64
vote_average            float64
vote_count              float64
genres                      str
production_companies        str
production_countries        str
adult                      bool
status                      str
genres_list                 str
primary_genre               str
release_year              int64
dtype: object
ratings
userId                      int64
movieId                     int64
rating                    float64
timestamp                   int64
rating_datetime    datetime64[us]
dtype: object
links
movieId      int64
imdbId       int64
tmdbId     float64
dtype: object
credits
tmdbId      int64
director    

In [8]:

for name, df in tables.items():
    print(f'\n{name}')
    display(df.dtypes.to_frame('dtype').head(20))
    display(df.isna().mean().sort_values(ascending=False).head(10).to_frame('missing_rate'))


movies


,dtype
movieId,int64
tmdbId,int64
imdb_id,str
title,str
original_title,str
original_language,str
release_date,str
runtime,float64
budget,int64
revenue,float64


,missing_rate
primary_genre,0.003854
genres_list,0.003854
status,0.000220
movieId,0.000000
vote_average,0.000000
adult,0.000000
production_countries,0.000000
production_companies,0.000000
genres,0.000000
vote_count,0.000000



ratings


,dtype
userId,int64
movieId,int64
rating,float64
timestamp,int64
rating_datetime,datetime64[us]


,missing_rate
userId,0.0
movieId,0.0
rating,0.0
timestamp,0.0
rating_datetime,0.0


... (5 outputs left out)

,missing_rate
top_cast,0.009910
director,0.002643
tmdbId,0.000000



keywords


,dtype
tmdbId,int64
keywords_list,str


,missing_rate
keywords_list,0.083792
tmdbId,0.000000


## Duplicates
 No duplicate keys were found in this data set. Each table was also checked for duplicated rows, none were present in all 5 tables.

In [9]:
for name, df in tables.items():
    print(name)
    print(df.duplicated().sum())
    print()

movies
0

ratings
0

links
0

credits
0

keywords
0



In [10]:
key_audit = pd.DataFrame([
    {"table": "movies", "expected_key": "movieId", "duplicate_keys": movies["movieId"].duplicated().sum(), "missing_key_values": movies["movieId"].isna().sum()},
    {"table": "movies", "expected_key": "tmdbId", "duplicate_keys": movies["tmdbId"].duplicated().sum(), "missing_key_values": movies["tmdbId"].isna().sum()},
    {"table": "ratings", "expected_key": "userId + movieId", "duplicate_keys": ratings[["userId", "movieId"]].duplicated().sum(), "missing_key_values": ratings[["userId", "movieId"]].isna().sum().sum()},
    {"table": "links", "expected_key": "movieId", "duplicate_keys": links["movieId"].duplicated().sum(), "missing_key_values": links["movieId"].isna().sum()},
    {"table": "credits", "expected_key": "tmdbId", "duplicate_keys": credits["tmdbId"].duplicated().sum(), "missing_key_values": credits["tmdbId"].isna().sum()},
    {"table": "keywords", "expected_key": "tmdbId", "duplicate_keys": keywords["tmdbId"].duplicated().sum(), "missing_key_values": keywords["tmdbId"].isna().sum()},
])

key_audit


,table,expected_key,duplicate_keys,missing_key_values
0,movies,movieId,0,0
1,movies,tmdbId,0,0
2,ratings,userId + movieId,0,0
3,links,movieId,0,0
4,credits,tmdbId,0,0
5,keywords,tmdbId,0,0


## Suspicious Values 
After analyzing all 5 tables, the movies table contained a few suspicious values, such as runtime minimum of 0 and maximum of 1140 minutes. A budget and revenue minimum of $0. These value are unrealistic and needs further investigation.  

In [11]:
for name, df in tables.items():
    print(name)
    print(df.describe())

movies
             movieId         tmdbId      runtime        budget       revenue  \
count    9082.000000    9082.000000  9082.000000  9.082000e+03  9.082000e+03   
mean    30963.741577   38710.463885   105.661418  1.664789e+07  4.918728e+07   
std     40659.690679   62123.411079    30.350712  3.345276e+07  1.301801e+08   
min         1.000000       2.000000     0.000000  0.000000e+00  0.000000e+00   
25%      2843.250000    9446.250000    93.000000  0.000000e+00  0.000000e+00   
50%      6265.500000   15775.000000   102.000000  1.000000e+05  2.716080e+05   
75%     56026.250000   39010.250000   115.000000  2.000000e+07  3.672528e+07   
max    163949.000000  416437.000000  1140.000000  3.800000e+08  2.787965e+09   

        popularity  vote_average    vote_count  release_year  
count  9082.000000   9082.000000   9082.000000   9082.000000  
mean      7.268905      6.362178    439.050870   1991.951993  
std       9.372314      1.038748    996.156492     19.387707  
min       0.000004  

## Initial Cleaning Plan and Target/Outcome
I plan to correct missing values in important movie characteristics before conducting any data analysis. I will also further investigate the suspicious values in columns such as runtime, budget, and revenue. Keeping the original raw data unchanged, some data in tables may need to be changed during the cleaning process. 
The analysis could examine how movie characteristics such as genre, runtime, director, cast, and keywords are related to movie ratings. 

## Ethics, Privacy, Governance
The ratings table contains user IDs, it is important that users are not identified or analyzed personally based on ratings. Any missing or suspicious values should be investigated and documented during the cleaning process. The original raw data should stay unchanged to keep the analysis organized and reproducible. 

## End of Milestone 1

## Milestone 2 Starter: Integration, Transformation, and Preliminary ABT

Build a preliminary ABT. State exactly what one row means, what keys define one row, and what tables were joined or aggregated.

Critical integration reminder:

Before creating a movie-level ABT, aggregate ratings. Examples include:

- average rating by `movieId`
- rating count by `movieId`
- rating standard deviation by `movieId`
- first and last rating date by `movieId`
- distinct user count by `movieId`

After merging, validate that `movieId` is not duplicated in the ABT.

### Updated Project Direction
Continuing from Milestone 1 data profiling, this project will investigate which movie characteristics are associated with higher ratings. The movies dataset contains movies, user ratings, credits, keywords. Since ratings are accounted for at the user-movie level, ratings will be summarized by movie id. The data profiling found missing genre, credit, keyword and tmdb values, and unrealsitic runtimes and budget/revenue values, these will be reviewed and documented during cleaning. The anaysis will describle patterns in the dataset, not clain a movie charactersic cased higher ratings.


### Data Cleaning and Transformation

In [ ]:

movies.shape, ratings.shape, links.shape, credits.shape, keywords.shape

((9082, 21), (99810, 5), (9125, 3), (9082, 3), (9082, 2))

In [13]:
movies_clean = movies.copy()
ratings_clean = ratings.copy()
links_clean = links.copy()
credits_clean = credits.copy()
keywords_clean = keywords.copy()


In [14]:
movies_clean.shape, ratings_clean.shape, links_clean.shape, credits_clean.shape, keywords_clean.shape

((9082, 21), (99810, 5), (9125, 3), (9082, 3), (9082, 2))

In [18]:
movies_clean.isna().sum()


movieId                  0
tmdbId                   0
imdb_id                  0
title                    0
original_title           0
original_language        0
release_date             0
runtime                  0
budget                   0
revenue                  0
popularity               0
vote_average             0
vote_count               0
genres                   0
production_companies     0
production_countries     0
adult                    0
status                   2
genres_list             35
primary_genre           35
release_year             0
dtype: int64

In [19]:
raw_movies= movies.copy()
movies_clean = movies.copy()

cleaning_log = []

print("raw data shape", raw_movies.shape)
print("working copy shape", movies_clean.shape)

raw data shape (9082, 21)
working copy shape (9082, 21)


In [17]:
print("Movies Table")
print(movies_clean.isna().sum())

print("Ratings Table")
print(ratings_clean.isna().sum())

print("Links Table")
print(links_clean.isna().sum())

print("Credits Table")
print(credits_clean.isna().sum())

print("Keywords Table")
print(keywords_clean.isna().sum())

Movies Table
movieId                  0
tmdbId                   0
imdb_id                  0
title                    0
original_title           0
original_language        0
release_date             0
runtime                  0
budget                   0
revenue                  0
popularity               0
vote_average             0
vote_count               0
genres                   0
production_companies     0
production_countries     0
adult                    0
status                   2
genres_list             35
primary_genre           35
release_year             0
dtype: int64
Ratings Table
userId             0
movieId            0
rating             0
timestamp          0
rating_datetime    0
dtype: int64
Links Table
movieId     0
imdbId      0
tmdbId     13
dtype: int64
Credits Table
tmdbId       0
director    24
top_cast    90
dtype: int64
Keywords Table
tmdbId             0
keywords_list    761
dtype: int64


In [18]:
movies_clean[["runtime","budget", "revenue"]].describe()

,runtime,budget,revenue
count,9082.000000,9.082000e+03,9.082000e+03
mean,105.661418,1.664789e+07,4.918728e+07
std,30.350712,3.345276e+07,1.301801e+08
min,0.000000,0.000000e+00,0.000000e+00
25%,93.000000,0.000000e+00,0.000000e+00
50%,102.000000,1.000000e+05,2.716080e+05
75%,115.000000,2.000000e+07,3.672528e+07
max,1140.000000,3.800000e+08,2.787965e+09


In [19]:
(movies_clean[["runtime", "budget", "revenue"]] == 0).sum()

runtime      20
budget     4443
revenue    4322
dtype: int64

In [21]:
movies_clean[
    (movies_clean["runtime"] == 0)
    | (movies_clean["budget"] == 0)
    | (movies_clean["revenue"] == 0)
][["title", "runtime", "budget", "revenue"]]

,title,runtime,budget,revenue
2,Grumpier Old Men,101.0,0,0.0
4,Father of the Bride Part II,106.0,0,76578911.0
6,Sabrina,127.0,58000000,0.0
7,Tom and Huck,97.0,0,0.0
11,Dracula: Dead and Loving It,88.0,0,0.0
...,...,...,...,...
9075,Body,75.0,0,0.0
9076,Sharknado 4: The 4th Awakens,85.0,0,0.0
9077,The Last Brickmaker in America,85.0,8000000,0.0
9078,Rustom,150.0,1000000,0.0


In [22]:
movies_clean["budget"] = movies_clean["budget"].replace(0, None)
movies_clean["revenue"] = movies_clean["revenue"].replace(0, None)

In [23]:
(movies_clean[["budget", "revenue"]] == 0).sum()

budget     0
revenue    0
dtype: int64

In [24]:
movies_clean[["budget", "revenue"]].isna().sum()

budget     4443
revenue    4322
dtype: int64

In [ ]:
movies_clean.loc[ movies_clean["runtime"] == 0,
                 ["title", "runtime"]
]

,title,runtime
200,Destiny Turns on the Radio,0.0
579,The Run of the Country,0.0
701,Nothing to Lose,0.0
710,1-900,0.0
920,Mina Tannenbaum,0.0
929,Dead Tired,0.0
1159,The Machine,0.0
1314,Wonderland,0.0
3119,Barenaked in America,0.0
4029,30 YEARS TO LIFE,0.0


In [26]:
movies_clean["runtime"] = movies_clean["runtime"].replace(0, None)

In [27]:
movies_clean["runtime"].isna().sum()

np.int64(20)

In [29]:
movies_clean["runtime"] = movies_clean["runtime"].astype(float)

In [30]:
movies_clean["runtime"].describe()

count    9062.000000
mean      105.894615
std        29.975022
min         2.000000
25%        93.000000
50%       102.000000
75%       115.000000
max      1140.000000
Name: runtime, dtype: float64

In [ ]:
movies_clean.loc[movies_clean["runtime"] == 1140,
    ["title", "release_date", "runtime"]
]

,title,release_date,runtime
8181,Jazz,2001-01-09,1140.0


In [32]:
movies_clean["runtime"] = movies_clean["runtime"].replace(1140, None).astype(float)

In [33]:
(movies_clean["runtime"]==1140).sum()

np.int64(0)

In [38]:
movies_clean["runtime"].describe()

count    9062.000000
mean      105.894615
std        29.975022
min         2.000000
25%        93.000000
50%       102.000000
75%       115.000000
max      1140.000000
Name: runtime, dtype: float64

In [36]:
movies_clean.loc[movies_clean["title"] == "Jazz","runtime"] = movies.loc[movies["title"] == "Jazz", "runtime"]

In [37]:
movies_clean.loc[movies_clean["title"] == "Jazz", "runtime"]

8181    1140.0
Name: runtime, dtype: float64

### Data cleaning update

I changed the zero values for budget and revenue to missing values because they are unreported values. I ke[t the movie rows. I also changed the 20 zero runtimes to missing values. I reviewed the unusual runtimes, but after discovering what movies they were, I kept them. No movie rows were deleted.

In [39]:
movies_clean["status"].value_counts(dropna=False)

status
Released           9061
Rumored              11
Post Production       7
NaN                   2
In Production         1
Name: count, dtype: int64

In [41]:
movies_clean["original_language"].value_counts(dropna=False)

original_language
en    7947
fr     272
ja     185
de     113
it      98
es      88
cn      47
sv      41
zh      39
ko      31
da      30
ru      27
pt      25
nl      16
fa      14
fi      13
hi      13
cs      10
tr       8
no       7
th       7
he       6
pl       6
el       5
sr       4
hu       4
bn       3
xx       3
id       3
vi       2
nb       2
ro       2
ar       2
bo       1
sk       1
bs       1
is       1
ps       1
uk       1
af       1
et       1
lo       1
Name: count, dtype: int64

In [42]:
movies_clean["primary_genre"].value_counts(dropna=False)

primary_genre
Drama              2413
Comedy             2168
Action             1177
Adventure           517
Horror              497
Crime               417
Documentary         383
Thriller            263
Fantasy             244
Animation           236
Romance             189
Science Fiction     126
Mystery              96
Music                82
Family               76
War                  52
Western              48
History              44
NaN                  35
TV Movie             18
Foreign               1
Name: count, dtype: int64

In [43]:
movies_clean["release_date"]=pd.to_datetime(movies_clean["release_date"],
errors="coerce",
format="mixed")
                                        

In [44]:
movies_clean["release_date"].dtype

dtype('<M8[us]')

In [45]:
movies_clean["release_date"].isna().sum()

np.int64(0)

In [50]:
movies_clean["release_date"].head()

0   1995-10-30
1   1995-12-15
2   1995-12-22
3   1995-12-22
4   1995-02-10
Name: release_date, dtype: datetime64[us]

In [51]:
movies_clean[["runtime","budget","revenue"]].dtypes

runtime    float64
budget      object
revenue     object
dtype: object

In [52]:
movies_clean["budget"]=pd.to_numeric(movies_clean["budget"], errors="coerce")
movies_clean["revenue"]=pd.to_numeric(movies_clean["revenue"], errors="coerce")

In [53]:
movies_clean[["runtime","budget","revenue"]].dtypes

runtime    float64
budget     float64
revenue    float64
dtype: object

In [54]:
movies_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 9082 entries, 0 to 9081
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   movieId               9082 non-null   int64         
 1   tmdbId                9082 non-null   int64         
 2   imdb_id               9082 non-null   str           
 3   title                 9082 non-null   str           
 4   original_title        9082 non-null   str           
 5   original_language     9082 non-null   str           
 6   release_date          9082 non-null   datetime64[us]
 7   runtime               9062 non-null   float64       
 8   budget                4639 non-null   float64       
 9   revenue               4760 non-null   float64       
 10  popularity            9082 non-null   float64       
 11  vote_average          9082 non-null   float64       
 12  vote_count            9082 non-null   float64       
 13  genres                9082 no

### Data cleaning update
I converted release_date from test to a date/time type, all realsed dated were converted successfully. budget, and revenue were all converted to numeric data types from object. 

In [57]:
ratings_clean.groupby("movieId")

In [58]:
rating_summary = (
    ratings_clean.groupby("movieId")
    .agg(
        user_rating_mean=("rating", "mean"),
        user_rating_count=("rating", "size"),
        user_rating_std=("rating", "std"),
        first_rating_date=("rating_datetime", "min"),
        last_rating_date=("rating_datetime", "max")
    )
    .reset_index()
)

abt = (
    movies_clean
    .merge(rating_summary, on="movieId", how="left", validate="one_to_one")
    .merge(credits_clean, on="tmdbId", how="left", validate="one_to_one")
    .merge(keywords_clean, on="tmdbId", how="left", validate="one_to_one")
)

abt["has_revenue"] = (pd.to_numeric(abt["revenue"], errors="coerce") > 0).astype("Int64")
abt["rating_target_eligible"] = abt["user_rating_count"].ge(20)
abt["high_user_rating"] = (
    abt["user_rating_mean"].ge(3.75)
    .where(abt["rating_target_eligible"])
    .astype("Int64")
)


In [59]:
abt_validation = pd.Series({
    "movie_rows_before_merges": len(movies),
    "abt_rows_after_merges": len(abt),
    "duplicate_movie_ids": abt["movieId"].duplicated().sum(),
    "duplicate_tmdb_ids": abt["tmdbId"].duplicated().sum(),
    "movies_with_rating_summary": abt["user_rating_count"].notna().sum(),
    "movies_eligible_for_rating_target": abt["high_user_rating"].notna().sum(),
})

display(abt_validation)
display(abt.head())
display(abt.isna().mean().sort_values(ascending=False).head(15).to_frame("missing_rate"))


movie_rows_before_merges             9082
abt_rows_after_merges                9082
duplicate_movie_ids                     0
duplicate_tmdb_ids                      0
movies_with_rating_summary           9025
movies_eligible_for_rating_target    1300
dtype: int64

,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year,user_rating_mean,user_rating_count,user_rating_std,first_rating_date,last_rating_date,director,top_cast,keywords_list,has_revenue,rating_target_eligible,high_user_rating
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000.0,373554033.0,21.946943,7.7,5415.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Comedy|Family,Animation,1995,3.872470,247.0,0.958981,1996-03-30 19:00:13,2016-10-06 19:55:11,John Lasseter,Tom Hanks|Tim Allen|Don Rickles,jealousy|toy|boy|friendship|friends|rivalry|bo...,1,True,1
1,2,8844,tt0113497,Jumanji,Jumanji,en,1995-12-15,104.0,65000000.0,262797249.0,17.015539,6.9,2413.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Fantasy|Family,Adventure,1995,3.401869,107.0,0.880714,1996-03-30 19:12:30,2016-08-01 17:42:33,Joe Johnston,Robin Williams|Jonathan Hyde|Kirsten Dunst,board game|disappearance|based on children's b...,1,True,0
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,101.0,NaN,NaN,11.712900,6.5,92.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Romance|Comedy,Romance,1995,3.161017,59.0,1.150115,1996-06-05 06:19:04,2016-08-16 22:07:21,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret,fishing|best friend|duringcreditsstinger|old men,0,True,0
3,4,31357,tt0114885,Waiting to Exhale,Waiting to Exhale,en,1995-12-22,127.0,16000000.0,81452156.0,3.859495,6.1,34.0,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",[{'name': 'Twentieth Century Fox Film Corporat...,"[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy|Drama|Romance,Comedy,1995,2.384615,13.0,0.938835,1996-06-10 16:45:35,2004-07-27 06:14:12,Forest Whitaker,Whitney Houston|Angela Bassett|Loretta Devine,based on novel|interracial relationship|single...,1,False,<NA>
4,5,11862,tt0113041,Father of the Bride Part II,Father of the Bride Part II,en,1995-02-10,106.0,NaN,76578911.0,8.387519,5.7,173.0,"[{'id': 35, 'name': 'Comedy'}]","[{'name': 'Sandollar Productions', 'id': 5842}...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy,Comedy,1995,3.267857,56.0,0.948512,1996-04-14 14:23:59,2016-08-16 22:15:47,Charles Shyer,Steve Martin|Diane Keaton|Martin Short,baby|midlife crisis|confidence|aging|daughter|...,1,True,0


,missing_rate
high_user_rating,0.856860
budget,0.489209
revenue,0.475886
user_rating_std,0.341224
keywords_list,0.083792
top_cast,0.009910
last_rating_date,0.006276
first_rating_date,0.006276
user_rating_count,0.006276
user_rating_mean,0.006276


### ABT Explanation
The abt has one row per movie. The primary key, movieId identifies each movie per row. tmdbId connects the movie row to credits, and keywords. left joined credits_clean, and keywords_clean ysing tmdbId. I did not join links table because movies_clean already contained it. the ABT has 9082 movies, and 9,025 rating summary. 1300 meet the minimum of 20 ratings target. Before and after joining movie counted reamined at 9082, and checked for duplicate values were 0. New columns were created such as  user_rating_mean, user_rating_count, user_rating_std, first_rating_date, and last_rating_date. Budget and revenue values are missing becasue original zero values were treated as unreported. Some movies lack keywords, cast, director, or genre information, and those values remain missing. No movies were lost during the join.

### SQL Validation Connected to the ABT

The project milestone asks for database or SQL work that supports the workflow. This in-memory SQLite check verifies the movie-level ABT row count and key uniqueness without creating another permanent database file.


In [62]:
%pip install duckdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 21.7 MB/s  0:00:00 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [64]:
import duckdb

duckdb.sql("""
select
    count(*) as abt_rows,
    count(distinct movieId) as unique_movie_ids,
    sum(case when high_user_rating is null then 1 else 0 end) as target_not_defined_rows
FROM abt
""").df()

,abt_rows,unique_movie_ids,target_not_defined_rows
0,9082,9082,7782.0


### ABT SQL EXPLANATION: 
This SQL query validated the ABT, it checked the structed of the table having 9082 rows, confirmed that each row was one movie using unique_movie_ids, and showed number of movies that did not meet the 20-rating minimum. 

In [65]:
duckdb.sql("""
select
    primary_genre, 
    count(*) as eligible_movie_count,
    round(avg(user_rating_mean), 2) as average_user_rating
from abt
where rating_target_eligible=true and primary_genre is not null
group by primary_genre
order by average_user_rating desc

""").df()


,primary_genre,eligible_movie_count,average_user_rating
0,Documentary,5,4.00
1,Western,5,3.98
2,War,5,3.98
3,Mystery,16,3.90
4,Drama,293,3.75
5,Crime,82,3.73
6,Animation,38,3.59
7,History,6,3.59
8,Family,12,3.54
9,Thriller,34,3.53


### genre SQL explanation
This SQL query help with the business question by grouping movings by primary genre, counting how many eligble movies each genre has, and calcualting thier average user ratings. It shows which genres have higher average user ratings. This gives a place to begin analysis, as it shows which genres have higher ratings, it does not measure popularity or show us that a specific genre causes higher user ratings. Looking at the table genres with higher average ratings have lower movie counts so it is important to take this into account as their avergaes may be less reliable. 

In [70]:
abt = abt[
    [
        "movieId", "title", "original_language", "release_year",
        "runtime", "budget", "revenue", "popularity", "genres_list",
        "primary_genre", "production_companies", "production_countries",
        "adult", "director", "top_cast", "keywords_list",
        "user_rating_mean", "user_rating_count", "user_rating_std",
        "rating_target_eligible", "high_user_rating"
    ]
]

In [71]:
abt.columns

Index(['movieId', 'title', 'original_language', 'release_year', 'runtime',
       'budget', 'revenue', 'popularity', 'genres_list', 'primary_genre',
       'production_companies', 'production_countries', 'adult', 'director',
       'top_cast', 'keywords_list', 'user_rating_mean', 'user_rating_count',
       'user_rating_std', 'rating_target_eligible', 'high_user_rating'],
      dtype='str')

### Data Dictionary Starter

Complete the text fields for every column you keep in the final ABT. Remove unused columns before finalizing the dictionary so the documentation and exported ABT stay aligned.


In [72]:
data_dictionary = pd.DataFrame({
    "column_name": abt.columns,
    "data_type": [str(abt[column].dtype) for column in abt.columns],
    "description": "",
    "source": "",
    "transformation_or_derivation": "",
    "notes_or_concerns": "",
})

# Hint: use .loc[] to document one column at a time, following the Week 11 pattern.
data_dictionary.head()


,column_name,data_type,description,source,transformation_or_derivation,notes_or_concerns
0,movieId,int64,,,,
1,title,str,,,,
2,original_language,str,,,,
3,release_year,int32,,,,
4,runtime,float64,,,,


In [76]:
doc_fields = [ "description", "source", "transformation_or_derivation","notes_or_concerns"]

example_entries = {
    "movieId": [
        "unique identifier for each movie.",
        "movies_metadata.csv",
        "No change; used as the primary key for each movie per row",
        "should be unique and non-missing"    ],
    "runtime" : [
        "The runtime of each movie in minutes.",
        "movies_metadata.csv",
        "changed zero values to missing",
        "20 movies had zero runtime"
    ],
    "budget" : [
        "The budget of each movie in US dollars.",
        "movies_metadata.csv",
        "changed zero values to missing",
        "some movies had zero budget and treated as unreported"
    ],
    "revenue" : [
        "The revenue of each movie in US dollars.",
        "movies_metadata.csv",
        "changed zero values to missing",
        "some movies had zero revenue and treated as unreported"
    ],
    "title" : [
        "The title of each movie.",
        "movies_metadata.csv",
        "No change",
        "used to identify movies"
    ],
    "original_language" : [
        "The original language of each movie.",
        "movies_metadata.csv",
        "No change",
        "stored as short language codes, ex: en or fr"
    ],
    "release_year" : [
        "The release year of each movie.",
        "movies_metadata.csv",
        "No change",
        "useful to analyze ratings over time"
    ],
    "popularity" : [
        "The popularity score of each movie.",
        "movies_metadata.csv",
        "No change",
        "higher values indicate more popular movies"
    ],
    "genres_list" : [
        "The list of genres for each movie.",
        "movies_metadata.csv",
        "No change",
        "missing for 35 movies"
    ], 
    "primary_genre" : [
        "The primary genre of each movie.",
        "movies_metadata.csv",
        "No change",
        "missing for 35 movies, represents only one genre when a movie may have several"
    ],
    "production_companies" : [
        "The production companies for each movie.",
        "movies_metadata.csv",
        "No change",
        "stored as a list of company names"
    ],
    "production_countries" : [
        "The production countries for each movie.",
        "movies_metadata.csv",
        "No change",
        "stored as a list of country names"
    ],
    "adult" : [
        "Whether the movie is adult content.",
        "movies_metadata.csv",
        "No change",
        "stored as a boolean value"
    ],
    "director" : [
        "The director of each movie.",
        "credits_curated.csv",
        "joined to movies using tmdbId",
        "missing for some movies"
    ],
    "top_cast" : [
        "The top cast members for each movie.",
        "credits_curated.csv",
        "joined to movies using tmdbId",
        "missing for some movies, multiple cast members stored together as text"
    ],
    "keywords_list" : [
        "The list of keywords for each movie.",
        "keywords_curated.csv",
        "joined to movies using tmdbId",
        "missing for some movies"
    ],
    "user_rating_mean" : [
        "The mean user rating for each movie.",
        "ratings.csv",
        "mean rating grouped by movieId",
        "missing when movie has no ratings"
    ],
    "user_rating_count" : [
        "The number of user ratings for each movie.",
        "ratings.csv",
        "count of ratings grouped by movieId",
        "missing when movie has no ratings"
    ],
    "user_rating_std" : [
        "The standard deviation of user ratings for each movie.",
        "ratings.csv",
        "standard deviation of ratings grouped by movieId",
        "missing when movie has fewer than 2 ratings"
    ],
    "rating_target_eligible" : [
        "Whether the movie is eligible for a rating target.",
        "ratings.csv",
        "True when user_rating_count is at least 20",
        "20 rating cutoff is the project's chosen minimum"
    ],
    "high_user_rating" : [
        "Whether the movie has a high user rating.",
        "ratings.csv",
        "True when user_rating_mean is at least 3.75",
        "undefined when movie has less than 20 ratings"
    ]
    
}
for abt_column, values in example_entries.items():
    data_dictionary.loc[
        data_dictionary["column_name"] == abt_column,
        doc_fields
    ] = values
data_dictionary

,column_name,data_type,description,source,transformation_or_derivation,notes_or_concerns
0,movieId,int64,unique identifier for each movie.,movies_metadata.csv,No change; used as the primary key for each mo...,should be unique and non-missing
1,title,str,The title of each movie.,movies_metadata.csv,No change,used to identify movies
2,original_language,str,The original language of each movie.,movies_metadata.csv,No change,"stored as short language codes, ex: en or fr"
3,release_year,int32,The release year of each movie.,movies_metadata.csv,No change,useful to analyze ratings over time
4,runtime,float64,The runtime of each movie in minutes.,movies_metadata.csv,changed zero values to missing,20 movies had zero runtime
5,budget,float64,The budget of each movie in US dollars.,movies_metadata.csv,changed zero values to missing,some movies had zero budget and treated as unr...
6,revenue,float64,The revenue of each movie in US dollars.,movies_metadata.csv,changed zero values to missing,some movies had zero revenue and treated as un...
7,popularity,float64,The popularity score of each movie.,movies_metadata.csv,No change,higher values indicate more popular movies
8,genres_list,str,The list of genres for each movie.,movies_metadata.csv,No change,missing for 35 movies
9,primary_genre,str,The primary genre of each movie.,movies_metadata.csv,No change,"missing for 35 movies, represents only one gen..."


## Milestone 3 Starter: Prepared Dataset and Initial Model Application

Choose a target/outcome and features. Start simple. Your first model should be correct and interpretable before it becomes complex.

Modeling reminder:

- If you model `high_user_rating`, use only rows where that nullable target is defined; these movies have at least 20 ratings.
- State when the prediction is meant to occur. For a pre-release question, exclude post-release fields such as revenue, popularity, vote count, rating summaries, and rating dates.
- Do not interpret a rating model as proving artistic quality or causal effects.


### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 3 written workspace

Use this cell to document the prepared dataset and initial model/analysis required by the Canvas Milestone 3 page.

- Target/outcome:
- Feature variables:
- Rows included/excluded and why:
- Train/test split or evaluation plan:
- Baseline model or analysis:
- Initial metrics/results:
- Interpretation limits:



In [ ]:
# Milestone 3 code workspace
# Suggested sequence:
# 1. State the prediction or analysis timing in markdown.
# 2. Filter to rows where your target is defined.
# 3. Select only features available at that time.
# 4. Check class balance or the target distribution.
# 5. Create a reproducible train/test split when modeling.
# 6. Fit a simple baseline and evaluate it on test data.


In [8]:
# Example placeholder. Replace with your chosen target and feature set.
# target = 'your_target_column'
# features = ['feature_1', 'feature_2']
# model_df = abt[[target] + features].dropna()

abt.describe(include='all').T.head(25)

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
movieId,9082.0,NaN,NaN,NaN,30963.741577,1.0,2843.25,6265.5,56026.25,163949.0,40659.690679
tmdbId,9082.0,NaN,NaN,NaN,38710.463885,2.0,9446.25,15775.0,39010.25,416437.0,62123.411079
imdb_id,9082,9082,tt0114709,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,9082,8809,Hamlet,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
original_title,9082,8841,Hamlet,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
original_language,9082,42,en,7947,NaN,NaN,NaN,NaN,NaN,NaN,NaN
release_date,9082,6002,1994-01-01,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
runtime,9082.0,NaN,NaN,NaN,105.661418,0.0,93.0,102.0,115.0,1140.0,30.350712
budget,9082.0,NaN,NaN,NaN,16647890.737833,0.0,0.0,100000.0,20000000.0,380000000.0,33452764.781378
revenue,9082.0,NaN,NaN,NaN,49187276.79586,0.0,0.0,271608.0,36725282.5,2787965087.0,130180082.298936


In [77]:
abt.to_csv("movies_preliminary_abt.csv", index=False)
data_dictionary.to_csv("movies_data_dictionary.csv", index=False)

## Final Project Starter: Interpretation, Ethics, and Recommendations

Use this section to connect your analysis back to a business or research decision.

Address:

- Main finding
- What the model/analysis can and cannot support
- Limitations
- Ethics or responsible use concerns
- 2-3 cautious recommendations

### Read the Canvas final submission page first

Before completing this section, read the Canvas final submission assignment page. The Canvas page is the authoritative checklist for the final notebook, HTML report, required files, and submission format. Use this section to organize your final evidence and interpretation.



### Final report written workspace

Use this cell to draft the final interpretation required by the Canvas final submission page.

- Main finding:
- Evidence supporting the finding:
- Business or research implication:
- Limitations:
- Ethics/responsible use concerns:
- Recommendations:
- What you would improve with more time:



In [ ]:
# Final project code workspace
# Save final datasets, figures, or tables here when they are ready.
# Recommended filenames:
# final_abt_path = OUTPUT_DIR / "movies_final_abt.csv"
# dictionary_path = OUTPUT_DIR / "movies_data_dictionary.csv"
# metrics_path = OUTPUT_DIR / "movies_model_metrics.csv"
# report_path = REPORTS_DIR / "movies_final_report.html"


In [9]:
# Save final deliverables only after your ABT and documentation are complete.
# abt.to_csv(OUTPUT_DIR / "movies_final_abt.csv", index=False)
# data_dictionary.to_csv(OUTPUT_DIR / "movies_data_dictionary.csv", index=False)

# Before final submission, restart the kernel, run all cells, and export the notebook to HTML.
